## Descripción

Este notebook transforma el historial de pagos de cuotas en variables agregadas por cliente (`SK_ID_CURR`). El objetivo es medir puntualidad, atrasos, pagos faltantes y cumplimiento monetario a partir de los pagos observados.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('../../data')
df = pd.read_parquet(DATA_PATH/'installments_payments.parquet')

## Construcción de features de pagos de cuotas

La función `build_installments_payment_features` crea variables a nivel fila y luego las agrega por cliente. Primero calcula retrasos, pagos observados, pagos faltantes, pagos tardíos, pagos a tiempo y ratios entre lo pagado y lo esperado. Después resume esas señales por `SK_ID_CURR`.

Está capturando:

- **Puntualidad:** tasas de pago tardío y pago a tiempo.
- **Severidad de atraso:** promedio y máximo de días vencidos.
- **Calidad del registro:** tasa de pagos faltantes.
- **Cumplimiento monetario:** ratio pagado/cuota, subpagos y sobrepagos.
- **Carga financiera:** montos promedio esperados y pagados.
- **Tipo/calendario de producto:** versión máxima de plan y proporción asociada a tarjeta de crédito.

Estas features resumen comportamiento de pago observado en una estructura compatible con modelos a nivel cliente. Separar puntualidad, severidad y cumplimiento monetario permite capturar dimensiones distintas del riesgo.


In [2]:

def build_installments_payment_features(df_inst):
    inst = df_inst.copy()

    id_col = "SK_ID_CURR"
    prev_id_col = "SK_ID_PREV"

    # VARIABLES DERIVADAS A NIVEL FILA

    # Diferencia entre fecha real de pago y fecha esperada.
    # Si es > 0, pagó tarde.
    # Si es <= 0, pagó a tiempo o antes.
    inst["inst_payment_delay_days"] = (
        inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]
    )

    # Pago faltante: no hay fecha real de pago o no hay monto pagado
    inst["inst_is_missing_payment"] = (
        inst["DAYS_ENTRY_PAYMENT"].isna() |
        inst["AMT_PAYMENT"].isna()
    ).astype(int)

    # Pago realmente observado
    inst["inst_is_paid_observed"] = (
        inst["DAYS_ENTRY_PAYMENT"].notna() &
        inst["AMT_PAYMENT"].notna()
    ).astype(int)

    # Pago tarde solo si hay pago observado y el delay es positivo
    inst["inst_is_late_payment"] = (
        inst["inst_is_paid_observed"].eq(1) &
        inst["inst_payment_delay_days"].gt(0)
    ).astype(int)

    # Pago a tiempo o antes solo si hay pago observado
    inst["inst_is_on_time_payment"] = (
        inst["inst_is_paid_observed"].eq(1) &
        inst["inst_payment_delay_days"].le(0)
    ).astype(int)

    # Días de atraso: solo cuenta atraso positivo
    inst["inst_days_past_due"] = inst["inst_payment_delay_days"].clip(lower=0)

    # Para promedio de atraso severo, dejamos NaN cuando no hubo atraso
    inst["inst_days_past_due_positive"] = np.where(
        inst["inst_is_late_payment"].eq(1),
        inst["inst_payment_delay_days"],
        np.nan
    )

    # Ratio pagado / cuota esperada
    inst["inst_payment_ratio"] = np.where(
        inst["AMT_INSTALMENT"].gt(0) & inst["AMT_PAYMENT"].notna(),
        inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"],
        np.nan
    )

    inst["inst_payment_ratio"] = (
        inst["inst_payment_ratio"]
        .replace([np.inf, -np.inf], np.nan)
    )

    # Pago parcial
    inst["inst_is_underpayment"] = (
        inst["AMT_PAYMENT"].notna() &
        inst["AMT_INSTALMENT"].notna() &
        inst["AMT_INSTALMENT"].gt(0) &
        inst["AMT_PAYMENT"].lt(inst["AMT_INSTALMENT"])
    ).astype(int)

    # Pago por encima de la cuota esperada
    inst["inst_is_overpayment"] = (
        inst["AMT_PAYMENT"].notna() &
        inst["AMT_INSTALMENT"].notna() &
        inst["AMT_INSTALMENT"].gt(0) &
        inst["AMT_PAYMENT"].gt(inst["AMT_INSTALMENT"])
    ).astype(int)

    # Versión 0 corresponde a tarjeta de crédito
    inst["inst_is_credit_card"] = (
        inst["NUM_INSTALMENT_VERSION"].eq(0)
    ).astype(int)


    # FEATURES AGREGADAS POR SK_ID_CURR

    features = (
        inst.groupby(id_col)
        .agg(
            # Volumen de historial de pagos
            inst_prev_credit_count=(prev_id_col, "nunique"),
            inst_total_installments_count=("NUM_INSTALMENT_NUMBER", "count"),

            # Comportamiento de mora
            inst_late_payment_rate=("inst_is_late_payment", "mean"),
            inst_days_past_due_mean=("inst_days_past_due_positive", "mean"),
            inst_days_past_due_max=("inst_days_past_due", "max"),
            inst_on_time_payment_rate=("inst_is_on_time_payment", "mean"),
            inst_missing_payment_rate=("inst_is_missing_payment", "mean"),

            # Cumplimiento monetario
            inst_payment_ratio_mean=("inst_payment_ratio", "mean"),
            inst_underpayment_rate=("inst_is_underpayment", "mean"),
            inst_overpayment_rate=("inst_is_overpayment", "mean"),

            # Carga financiera
            inst_amt_instalment_mean=("AMT_INSTALMENT", "mean"),
            inst_amt_payment_mean=("AMT_PAYMENT", "mean"),

            # Cambios de calendario / tipo de producto
            inst_calendar_version_max=("NUM_INSTALMENT_VERSION", "max"),
            inst_credit_card_rate=("inst_is_credit_card", "mean")
        )
    )

    # TOTAL PAYMENT RATIO
    # suma pagada / suma esperada

    total_amounts = (
        inst.groupby(id_col)
        .agg(
            total_amt_payment=("AMT_PAYMENT", "sum"),
            total_amt_instalment=("AMT_INSTALMENT", "sum")
        )
    )

    features["inst_total_payment_ratio"] = (
        total_amounts["total_amt_payment"] /
        total_amounts["total_amt_instalment"].replace(0, np.nan)
    )


    # LIMPIEZA FINAL

    final_cols = [
        "inst_prev_credit_count",
        "inst_total_installments_count",

        "inst_late_payment_rate",
        "inst_days_past_due_mean",
        "inst_days_past_due_max",
        "inst_on_time_payment_rate",
        "inst_missing_payment_rate",

        "inst_payment_ratio_mean",
        "inst_underpayment_rate",
        "inst_overpayment_rate",
        "inst_total_payment_ratio",

        "inst_amt_instalment_mean",
        "inst_amt_payment_mean",

        "inst_calendar_version_max",
        "inst_credit_card_rate"
    ]

    features = (
        features
        .reindex(columns=final_cols)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .reset_index()
    )

    return features


inst_features = build_installments_payment_features(df)

display(inst_features.head())
print(inst_features.shape)
print(inst_features.columns.tolist())

,SK_ID_CURR,inst_prev_credit_count,inst_total_installments_count,inst_late_payment_rate,inst_days_past_due_mean,inst_days_past_due_max,inst_on_time_payment_rate,inst_missing_payment_rate,inst_payment_ratio_mean,inst_underpayment_rate,inst_overpayment_rate,inst_total_payment_ratio,inst_amt_instalment_mean,inst_amt_payment_mean,inst_calendar_version_max,inst_credit_card_rate
0,100001,2,7,0.142857,11.0,11.0,0.857143,0.0,1.0,0.0,0.0,1.0,5885.132143,5885.132143,2.0,0.0
1,100002,1,19,0.000000,0.0,0.0,1.000000,0.0,1.0,0.0,0.0,1.0,11559.247105,11559.247105,2.0,0.0
2,100003,3,25,0.000000,0.0,0.0,1.000000,0.0,1.0,0.0,0.0,1.0,64754.586000,64754.586000,2.0,0.0
3,100004,1,3,0.000000,0.0,0.0,1.000000,0.0,1.0,0.0,0.0,1.0,7096.155000,7096.155000,2.0,0.0
4,100005,1,9,0.111111,1.0,1.0,0.888889,0.0,1.0,0.0,0.0,1.0,6240.205000,6240.205000,2.0,0.0


(339587, 16)
['SK_ID_CURR', 'inst_prev_credit_count', 'inst_total_installments_count', 'inst_late_payment_rate', 'inst_days_past_due_mean', 'inst_days_past_due_max', 'inst_on_time_payment_rate', 'inst_missing_payment_rate', 'inst_payment_ratio_mean', 'inst_underpayment_rate', 'inst_overpayment_rate', 'inst_total_payment_ratio', 'inst_amt_instalment_mean', 'inst_amt_payment_mean', 'inst_calendar_version_max', 'inst_credit_card_rate']
